In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install py7zr

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import os
import torch.optim as optim
import matplotlib.pyplot as plt


# Data 

In [ ]:
sub_csv = pd.read_csv("/kaggle/input/competitions/cifar-10/sampleSubmission.csv")
sub_csv.head(5)

In [ ]:
train_labels = pd.read_csv("/kaggle/input/competitions/cifar-10/trainLabels.csv")
train_labels.head(5)

In [ ]:
button = True
if button == True:
    import py7zr
    with py7zr.SevenZipFile('/kaggle/input/competitions/cifar-10/train.7z', mode='r') as archive:
        archive.extractall(path='/kaggle/working/')
    with py7zr.SevenZipFile('/kaggle/input/competitions/cifar-10/test.7z', mode='r') as archive:
        archive.extractall(path='/kaggle/working/')

In [ ]:
train_path = "/kaggle/working/train"
test_path = "/kaggle/working/test"

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.5),
    transforms.ToTensor()
])
# No need to Resize because ViT can handle the variety of size img
test_transforms = transforms.Compose([
    transforms.ToTensor()
])

In [ ]:
list_labels = 'airplane automobile bird cat deer dog frog horse ship truck'
idx_to_label = {i : label for i,label in enumerate(list_labels.split(" "))}
label_to_idx = {label : i for i,label in idx_to_label.items()}
num_classes = len(label_to_idx)
label_to_idx

In [ ]:
from torch.utils.data import Dataset,DataLoader

class CIFAR10_Dataset(Dataset):
    def __init__(self, path, transforms, csv_file=None):
        # 1. Sắp xếp danh sách file để đảm bảo tính nhất quán
        self.list_img_dir = sorted(os.listdir(path))
        self.path = path
        self.transforms = transforms
        
        self.label_dict = None
        if csv_file is not None:
            self.label_dict = dict(zip(csv_file['id'], csv_file['label']))

    def __len__(self):
        return len(self.list_img_dir)

    def __getitem__(self, idx):
        # Lấy tên file ảnh hiện tại
        img_name = self.list_img_dir[idx]
        img_dir = os.path.join(self.path, img_name)
        idx_label = int(img_name.split('.')[0])
        # Đọc và biến đổi ảnh
        img = Image.open(img_dir).convert("RGB")
        img = self.transforms(img)
        
        # Trả về dữ liệu
        if self.label_dict is None:
            return img
        else:
            label_string = self.label_dict[idx_label]
            labels = torch.tensor(label_to_idx[label_string])
            return img, labels

In [ ]:
from torch.utils.data import random_split
train_val_dataset = CIFAR10_Dataset(train_path,train_transforms,train_labels)
test_dataset = CIFAR10_Dataset(test_path,test_transforms,None)
# train_loader = DataLoader(train_dataset,
#                           batch_size = 32,
#                           shuffle = True,
#                          )
test_loader = DataLoader(test_dataset,
                        batch_size = 32,
                        shuffle = False)

In [ ]:
train_length = int(0.9 * len(train_val_dataset))
train_dataset,val_dataset = random_split(train_val_dataset,[train_length,len(train_val_dataset)-train_length])
train_loader = DataLoader(train_dataset,
                        batch_size = 32,
                        shuffle = True)
val_loader = DataLoader(val_dataset,
                       batch_size = 32,
                       shuffle = False)

In [ ]:
train_length

In [ ]:
def show_image(loader,idx):
    mau = next(iter(loader))
    img = mau[0][idx].permute(1, 2, 0).tolist()
    label = mau[1][idx].tolist()
    plt.imshow(img)
    plt.title(f"The prediction for this image is {idx_to_label[label]}")
    plt.axis('off')
    plt.show()

In [ ]:
show_image(val_loader,0)

# ViT

## 1)Patchifying the Img

In [ ]:
from einops import rearrange
class Patch_Layer(nn.Module):
    def __init__(self,patch_size):
        super(Patch_Layer,self).__init__()
        self.patch_size = patch_size
    def forward(self,X):# Shape = B H W C
        patches = rearrange(X,
                           "... c (h p1) (w p2)-> ... (h w) (p1 p2 c) ",
                           p1 = self.patch_size,
                           p2 = self.patch_size)
        return patches

## 2)Patch Emb

bias = True

In [ ]:
class Patch_Emb(nn.Module):
    def __init__(self,patch_size,out_feat):
        super(Patch_Emb,self).__init__()
        self.C = 3 # THay doi duoc 
        self.Layer = nn.Linear(in_features = self.C * patch_size * patch_size,
                              out_features = out_feat,
                              bias = True) # NOTE LAI bias = TRUE
        self.patch_layer = Patch_Layer(patch_size)
    def forward(self,X):
        Patches = self.patch_layer(X) # (B,H,W,C) -> (B,H/p * W/p,p^2 * C)
        emb = self.Layer(Patches) # -> (B,H/p * W/p,emb_dims)
        return emb

## 3) Positional Emb

### Intergrate the inference mode 

In [ ]:
class Postional_Emb(nn.Module): # Tao parameters co cls roi nhe
    def __init__(self,patch_size,embed_dim):
        super(Postional_Emb,self).__init__()
        self.num_patches = 224 // patch_size
        self.Layer = nn.Parameter(torch.randn(1, int(self.num_patches** 2) + 1, embed_dim))
    # def forward(self,X,inference = False): # X shape = (B,197,emb_dim) Voi 197 = 1 + 3 * patch_size ** 2
    #     output = self.Layer
    #     if inference:
    #         output = F.interpolate(self.Layer,
    #                                size = X.shape[1:],
    #                                mode = "bicubic",
    #                                align_corners=False)
    #     else:
    #         return output + X
    def forward(self, X, inference = False):
        output = self.Layer
        if inference:
            pos_embed = self.Layer.unsqueeze(0) if self.Layer.dim() == 3 else self.Layer.view(1, 1, -1, 768)
            
            output = F.interpolate(
                pos_embed, 
                size=X.shape[1:],
                mode="bicubic", 
                align_corners=False
            )
            
            output = output.squeeze(0).squeeze(0)
            
        return output + X

## 4) Encoder Block 

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,num_heads,bias = False):
        super(MultiHeadAttention,self).__init__()
        assert d_out % num_heads == 0
        self.d_in = d_in
        self.d_out = d_out
        self.dim_heads = d_out // num_heads
        self.num_heads = num_heads
        self.Q = nn.Linear(d_in,d_out,bias = bias)
        self.K = nn.Linear(d_in,d_out,bias = bias)
        self.V = nn.Linear(d_in,d_out,bias = bias)
        self.out_proj = nn.Linear(d_out,d_out,bias = bias)
        self.norm = nn.LayerNorm(d_in)
    def forward(self,X): # b,num_patches ,emb_dim
        X = self.norm(X)
        
        b,num_patches, _ = X.shape
        Query = self.Q(X)
        Key = self.K(X)
        Value = self.V(X) # b,num_patches,d_out
        Query = Query.view(b,num_patches,self.num_heads,self.dim_heads) # (B,P,n_heads,dim_head)
        Key = Key.view(b,num_patches,self.num_heads,self.dim_heads)
        Value = Value.view(b,num_patches,self.num_heads,self.dim_heads)

        Query = Query.transpose(1,2) # (B,n_heads,P,dim_heads)
        Key = Key.transpose(1,2)
        Value = Value.transpose(1,2)

        attn_scores = Query @ Key.transpose(2,3) # (B,n_heads,P,P)
        context_vec = torch.softmax(attn_scores / self.dim_heads ** 0.5,dim = - 1) @ Value # (B,n_heads,P,dim_heads)
        context_vec = context_vec.transpose(1,2)
        # The same technique, concat.
        context_vec = context_vec.contiguous().view(b,num_patches,self.d_out)

        context_vec = self.out_proj(context_vec)
        return context_vec

In [ ]:
class FeedForward(nn.Module):
    def __init__(self,d_in,d_out,hidden_dims):
        super(FeedForward,self).__init__()
        self.out = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in,hidden_dims,bias = True),
            nn.GELU(),
            nn.Linear(hidden_dims,d_out,bias = True)
        )
    def forward(self,X):
        return self.out(X)

In [ ]:
class Encoder_Block(nn.Module):
    def __init__(self,CONFIG):
        super(Encoder_Block,self).__init__()
        self.Attention = MultiHeadAttention(CONFIG['emb_dims'],CONFIG['emb_dims'],CONFIG['num_heads'])
        self.ff = FeedForward(CONFIG['emb_dims'],CONFIG['emb_dims'],CONFIG['hidden_dimss'])
    def forward(self,X):
        X = self.Attention(X) + X
        return self.ff(X) + X

## ViT Architecture

H/p * W/p --> patch_length

In [ ]:
class ViT(nn.Module):
    def __init__(self,CONFIG):
        super(ViT,self).__init__()
        self.Patch_layer = Patch_Emb(CONFIG['patch_size'],CONFIG['emb_dims'])
        self.positional_emb = Postional_Emb(CONFIG['patch_size'],CONFIG['emb_dims'])
        self.ln_pre = nn.LayerNorm(CONFIG['emb_dims'])
        self.transformer_encoder = nn.ModuleList(Encoder_Block(CONFIG) for _ in range(CONFIG['layers']))
        self.ln_post = nn.LayerNorm(CONFIG['emb_dims'])
        self.mlp_head = nn.Linear(CONFIG['emb_dims'],CONFIG['num_classes'])
        self.cls = nn.Parameter(torch.rand(1, 1, CONFIG['emb_dims']))
    def forward(self,X,inference = False): # b,224,224,3
        b= X.shape[0]
        patched_X = self.Patch_layer(X) # (B,H/p * W/p,emb_dims) 
        cls_tokens = self.cls.expand(b,-1,-1)
        added_X = torch.concat([cls_tokens,patched_X],dim = 1) # B, patch_length + 1,emb_dims
        output = self.positional_emb(added_X,inference)
        output = self.ln_pre(output)
        for layer in self.transformer_encoder:
            output = layer(output)
        output = output[:,0,:]
        output = self.ln_post(output)
        output = self.mlp_head(output)
        return output # B,1,num_classes

# CONFIG

In [ ]:
CONFIG_ViT = {
'emb_dims' : 768,'hidden_dimss' : 3072,
'patch_size' : 32, 'dropout' : 0.4,
'num_heads' : 12,'layers' : 12,
'num_classes' : num_classes
}
LR = 1e-3
EPOCHS = 30

In [ ]:
model = ViT(CONFIG_ViT)

In [ ]:
img = torch.randn(1, 3, 288, 288)
a = model(img,inference = True)
print(a.shape)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
total_params

# Training Time !!!

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()
optimizers = optim.AdamW(model.parameters(),lr = LR,betas = [0.9,0.999],weight_decay = 0.03)

In [ ]:
def train_fn(train_loader,model,criterion,optimizers,DEVICE,num_iter_for_print = 50):
    model.train()
    avg_loss = 0
    for i,(inputs,targets) in enumerate(train_loader):
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)
        outputs = model(inputs)
        loss = criterion(outputs,targets)
        optimizers.zero_grad()
        loss.backward()
        optimizers.step()
        avg_loss += loss.item() / len(train_loader)
        if((i + 1) % num_iter_for_print == 0):
            print(f"Loss at iter {i+1} : {loss.item()} ")
    return avg_loss
def valid_fn(val_loader,model,criterion,DEVICE):
    model.eval()
    with torch.no_grad():
        avg_loss = 0
        for (inputs,targets) in val_loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs,targets)
            avg_loss += loss / len(val_loader)
    return avg_loss
def accuracy(loader,model,DEVICE):
    model.eval()
    with torch.no_grad():
        corrected_pred = 0
        total_pred = 0
        for (inputs,targets) in loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)   
            
            outputs = model(inputs)
            outputs = torch.argmax(outputs,dim = -1)
            batch_size = inputs.shape[0]
            corrected_pred += (outputs == targets).sum().item()
            total_pred += len(outputs)
    return corrected_pred / total_pred

def train(train_loader,val_loader,model,criterion,optimizers,EPOCHS,DEVICE,MODEL_FILE,num_iter_for_print = 50):
    train_loss_his = []
    val_loss_his = []
    train_acc_his = []
    val_acc_his = []
    best_val_acc = 0
    model.to(DEVICE)
    for epoch in range(1,EPOCHS + 1):
        avg_train_loss = train_fn(train_loader,model,criterion,optimizers,DEVICE,num_iter_for_print)
        avg_val_loss = valid_fn(val_loader,model,criterion,DEVICE)
        train_acc = accuracy(train_loader,model,DEVICE)
        val_acc = accuracy(val_loader,model,DEVICE)
        print(f"Train loss and Valid loss at epoch {epoch + 1}: {avg_train_loss} ------ {avg_val_loss}")
        print(f"Accuracy on Train and Valid at epoch {epoch + 1}: {train_acc} ------ {val_acc} ")
        train_loss_his.append(avg_train_loss)
        val_loss_his.append(avg_val_loss)
        train_acc_his.append(train_acc)
        val_acc_his.append(val_acc)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(),MODEL_FILE)
    return train_loss_his,val_loss_his,train_acc_his,val_acc_his

# Chú ý sửa val_loader

In [ ]:
MODEL_FILE = "model.pth"
train_loss_his,val_loss_his,train_acc_his,val_acc_his = train(train_loader,val_loader,model,criterion,optimizers,EPOCHS,DEVICE,MODEL_FILE,num_iter_for_print = 100)

# Inference time

In [ ]:
def prediction(img_dir,model):
    img = Image.open(img_dir).convert("RGB")
    transformed_img = test_transforms(img).unsqueeze(0) # 1x3x224x224
    DEVICE = next(model.parameters()).device
    transformed_img = transformed_img.to(DEVICE)
    model.eval()
    with torch.no_grad():
        output = model(transformed_img,inference = True)
    label = torch.argmax(output,dim = -1)
    label = idx_to_label[label.item()]
    plt.imshow(img)
    plt.title(f"The prediction for this image is {label}")
    plt.axis('off')
    plt.show()

In [ ]:
list_img_dir = os.listdir(test_path)
img_dir = os.path.join(test_path,list_img_dir[2])
prediction(img_dir,model)

# submission()

In [ ]:
model.eval()
with torch.no_grad():
    list_labels = []
    for inputs in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs, inference=True)
        predicted_indices = torch.argmax(outputs, dim=-1).cpu().numpy() 
        # 2. Dùng .extend() thay vì .append() để thêm các phần tử vào list
        list_labels.extend(idx_to_label[idx] for idx in predicted_indices)
submission = pd.DataFrame({
    "id": np.arange(1, len(list_labels) + 1),
    "label": list_labels
})
output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)